# M6a · Build `marts.mart_device_monthly_behavior`

**Goal:** run `sql/20_mart_device_monthly_behavior.sql` for every month that got data during the fact backfill.

After a clean backfill we don't yet know the exact list of touched months, so we pass the full distinct month list computed from `warehouse.fact_trip`.

**Exit criterion:** `SELECT COUNT(*) FROM marts.mart_device_monthly_behavior > 0` and the 35 feature columns have plausible distributions.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    months = pd.read_sql(text("""
        SELECT DISTINCT TO_CHAR(begin_path_time, 'YYYY-MM') AS ym
        FROM warehouse.fact_trip ORDER BY ym
    """), conn)['ym'].tolist()
print(f"{len(months)} distinct months to recompute")
print(months[:12], '...' if len(months) > 12 else '')


## 3. Execute

In [ ]:
import time
from accent_fleet.db import run_sql_file
from accent_fleet.pipeline.run_log import begin_run, end_run
run_id = begin_run(mode='notebook:mart_device_monthly_behavior')
t0 = time.time()
try:
    with transaction() as conn:
        result = run_sql_file(conn, '20_mart_device_monthly_behavior.sql',
                              params={'touched_months': months, 'etl_run_id': run_id})
    rows = result.rowcount or 0
    end_run(run_id, status='success', rows_loaded=rows)
    print(f"mart recomputed: {rows} rows in {time.time()-t0:.1f}s")
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc))
    raise


## 4. Inspect

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM marts.mart_device_monthly_behavior")).scalar_one()
    sample = pd.read_sql(text("SELECT * FROM marts.mart_device_monthly_behavior ORDER BY year_month DESC LIMIT 10"), conn)
    stats = pd.read_sql(text("""
      SELECT
        AVG(total_trips) avg_trips,
        AVG(total_distance_km) avg_distance,
        AVG(overspeed_count) avg_overspeed,
        AVG(night_trip_ratio) avg_night_ratio
      FROM marts.mart_device_monthly_behavior
    """), conn)
print(f"mart rows: {total:,}")
display(stats); display(sample)
assert total > 0, 'mart is empty'
